In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import adjusted_rand_score

# Robust resolution whether executed from notebooks/, repo root, or in-place.
_name = "derived_8.4-routing-optimize-1.0"
_candidates = [Path.cwd() / "notebooks" / "experiment" / _name,
              Path.cwd() / "experiment" / _name,
              Path.cwd()]
EXP_DIR = next((p for p in _candidates if (p / "config.yaml").exists()), Path.cwd())
sys.path.insert(0, str(EXP_DIR))
PROJECT_ROOT = EXP_DIR.parents[2]  # <exp> -> experiment -> notebooks -> repo root

from routing_opt.agreement import corrected_agreement, k_sweep_quality, station_purity
from routing_opt.data import load_experiment_data
from routing_opt.routers import get_router

with open(EXP_DIR / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
data = load_experiment_data(PROJECT_ROOT, config)
B54, V0 = data.shared_backbone_54, data.v0_features
SEED, TAU = int(config["router"]["seed"]), float(config["router"]["tau"])
print(f"EXP_DIR={EXP_DIR} PROJECT_ROOT={PROJECT_ROOT}")
print(f"train={len(data.train)} val={len(data.val)} test={len(data.test)} trainval={len(data.trainval)}")
print(f"V0={len(V0)} backbone54={len(B54)} overlap={len(set(V0) & set(B54))}")
print("V0-only:", sorted(set(V0) - set(B54)))
print("B54-only:", sorted(set(B54) - set(V0)))


EXP_DIR=/scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/derived_8.4-routing-optimize-1.0 PROJECT_ROOT=/scratch/group/p.cis250607.000/MDR-Project
train=9803 val=4805 test=6620 trainval=14608
V0=50 backbone54=54 overlap=17
V0-only: ['A_d_E_SAR_diff_kobs30', 'A_d_LST_modis_kobs14', 'A_d_LST_modis_kobs30', 'A_d_SMAP_sm_interp_kobs5', 'A_d_s2_b11_kobs30', 'A_grad_LST_modis_kobs14', 'A_grad_LST_modis_kobs30', 'A_grad_SMAP_sm_interp_kobs30', 'C_lag_F_NDMI_kobs30', 'D_sa_F_NDMI', 'D_sa_LST_modis', 'D_z_E_SAR_ratio', 'F_MSI', 'J_bio_bio15', 'J_clay_wfrac_b100', 'K_aspect_cos', 'K_aspect_sin', 'V_ema_G_API_kobs30', 'V_ema_SMAP_sm_interp_kobs30', 'V_rollmax_G_API_kobs30', 'V_rollmax_G_API_kobs7', 'V_rollmean_F_NDMI_kobs30', 'V_rollmin_G_API_kobs14', 'V_rollrng_E_SAR_ratio_kobs30', 'V_rollrng_F_NDVI_kobs14', 'V_rollrng_F_NDVI_kobs30', 'V_rollrng_G_API_kobs30', 'V_rollrng_LST_modis_kobs30', 'aspect', 'latitude', 'lia_std_asc_deg', 's2_b11', 'slope']
B54-only: ['A_d_E_SAR_ratio_kobs3

# derived_8.4-routing-optimize-1.0 — Guarded backbone routing diagnostics

This notebook is the sole generator of the R1–R4 tables in `README.md`. It fits KMeans routers on trainval only (CPU, no XGBoost training) and reports feature overlap, full-trainval agreement, per-LOSO-fold agreement with fallback rates, and a trainval-only K-sweep. Fitting never touches the test split except via `predict` (out-of-sample confirmation).


## R2 — Full-trainval partitions
Fits all four routers once on the full 7-station trainval and compares group sizes, station purity, and pairwise ARI/agreement on trainval and test. Expectation from gating-analysis-1.0: V0 vs Backbone ARI = 1.0 with sizes 10624/3984; GuardedV-A must reproduce the Backbone partition on known stations.


In [2]:
routers = {
    "V0_Full": get_router("Clustering_V0_Full_k2", backbone_54=B54, v0_features=V0, seed=SEED, tau=TAU),
    "Backbone": get_router("Clustering_Backbone54_k2", backbone_54=B54, seed=SEED, tau=TAU),
    "GuardedV-A": get_router("Guarded_Backbone54_k2", backbone_54=B54, seed=SEED, tau=TAU),
    "StationMeanV-B": get_router("StationMean_Backbone54_k2", backbone_54=B54, seed=SEED, tau=TAU),
}
for r in routers.values():
    r.fit(data.trainval)
print("label_flip:", {n: r.label_flip_applied for n, r in routers.items()})
print("margin_p5:", {n: round(r.margin_p5, 4) for n, r in routers.items()})

def _labs(router, frame):
    return np.asarray(router.predict(frame)).ravel().astype(int)

names = list(routers)
for fname, frame in [("trainval", data.trainval), ("test", data.test)]:
    fr = frame.reset_index(drop=True)
    labs = {n: _labs(routers[n], frame) for n in names}
    print(f"--- {fname} n={len(frame)} ---")
    for n in names:
        sizes = np.bincount(labs[n]).tolist()
        pur = station_purity(fr, labs[n])["purity"].mean()
        print(f"{n:14s} sizes={sizes} mean_purity={pur:.4f}")
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            print(f"ARI {a} vs {b}: {adjusted_rand_score(labs[a], labs[b]):.4f} "
                  f"agree={corrected_agreement(labs[a], labs[b]):.4f}")


label_flip: {'V0_Full': False, 'Backbone': False, 'GuardedV-A': False, 'StationMeanV-B': False}
margin_p5: {'V0_Full': 1.956, 'Backbone': 1.5481, 'GuardedV-A': 1.5481, 'StationMeanV-B': 1.5481}
--- trainval n=14608 ---
V0_Full        sizes=[10624, 3984] mean_purity=1.0000
Backbone       sizes=[10624, 3984] mean_purity=1.0000
GuardedV-A     sizes=[10624, 3984] mean_purity=1.0000
StationMeanV-B sizes=[10624, 3984] mean_purity=1.0000


ARI V0_Full vs Backbone: 1.0000 agree=1.0000
ARI V0_Full vs GuardedV-A: 1.0000 agree=1.0000
ARI V0_Full vs StationMeanV-B: 1.0000 agree=1.0000
ARI Backbone vs GuardedV-A: 1.0000 agree=1.0000
ARI Backbone vs StationMeanV-B: 1.0000 agree=1.0000
ARI GuardedV-A vs StationMeanV-B: 1.0000 agree=1.0000
--- test n=6620 ---
V0_Full        sizes=[4817, 1803] mean_purity=1.0000
Backbone       sizes=[4817, 1803] mean_purity=0.9997
GuardedV-A     sizes=[4817, 1803] mean_purity=1.0000
StationMeanV-B sizes=[4817, 1803] mean_purity=1.0000
ARI V0_Full vs Backbone: 0.9987 agree=0.9997
ARI V0_Full vs GuardedV-A: 1.0000 agree=1.0000
ARI V0_Full vs StationMeanV-B: 1.0000 agree=1.0000
ARI Backbone vs GuardedV-A: 0.9987 agree=0.9997
ARI Backbone vs StationMeanV-B: 0.9987 agree=0.9997
ARI GuardedV-A vs StationMeanV-B: 1.0000 agree=1.0000


## R3 — Per-LOSO-fold agreement and fallback rates
Refits every router on each 6-station fold trainval (held-out station unseen by construction) and reports pairwise ARI on fold-trainval and the held-out test rows, plus the GuardedV-A fallback-mask rate on the held-out rows. This is where V0 vs Backbone diverge despite ARI = 1.0 on the full trainval.


In [3]:
STRATEGY_OF = {"V0_Full": "Clustering_V0_Full_k2", "Backbone": "Clustering_Backbone54_k2",
               "GuardedV-A": "Guarded_Backbone54_k2", "StationMeanV-B": "StationMean_Backbone54_k2"}
fold_rows = []
for station in sorted(data.test["station_id"].unique()):
    tr = data.trainval[data.trainval["station_id"] != station].reset_index(drop=True)
    te = data.test[data.test["station_id"] == station].reset_index(drop=True)
    fr = {n: get_router(STRATEGY_OF[n], backbone_54=B54, v0_features=V0, seed=SEED, tau=TAU).fit(tr) for n in names}
    ltr = {n: _labs(fr[n], tr) for n in names}
    lte = {n: _labs(fr[n], te) for n in names}
    row = {"held_out": station}
    for a, b in [("V0_Full", "Backbone"), ("Backbone", "GuardedV-A"),
                 ("Backbone", "StationMeanV-B"), ("GuardedV-A", "StationMeanV-B")]:
        row[f"ARI_tr_{a}_vs_{b}"] = round(float(adjusted_rand_score(ltr[a], ltr[b])), 4)
        row[f"ARI_te_{a}_vs_{b}"] = round(float(adjusted_rand_score(lte[a], lte[b])), 4)
    row["fallback_rate_te"] = round(float(fr["GuardedV-A"].fallback_mask(te).mean()), 4)
    row["n_te"] = len(te)
    fold_rows.append(row)
print(pd.DataFrame(fold_rows).to_markdown(index=False))


| held_out              |   ARI_tr_V0_Full_vs_Backbone |   ARI_te_V0_Full_vs_Backbone |   ARI_tr_Backbone_vs_GuardedV-A |   ARI_te_Backbone_vs_GuardedV-A |   ARI_tr_Backbone_vs_StationMeanV-B |   ARI_te_Backbone_vs_StationMeanV-B |   ARI_tr_GuardedV-A_vs_StationMeanV-B |   ARI_te_GuardedV-A_vs_StationMeanV-B |   fallback_rate_te |   n_te |
|:----------------------|-----------------------------:|-----------------------------:|--------------------------------:|--------------------------------:|------------------------------------:|------------------------------------:|--------------------------------------:|--------------------------------------:|-------------------:|-------:|
| BeaverPass_WA_990     |                       1      |                            1 |                          1      |                               1 |                              1      |                                   1 |                                1      |                                     1 |     

## R4 — K-sweep quality, trainval-only
Internal indices (silhouette, Calinski-Harabasz, Davies-Bouldin) plus mean station purity and minimum cluster share for K in {2, 3, 4} on the 54 backbone, computed on trainval only. Test purity is out-of-sample confirmation, never a selection criterion. This table is the automatic K-selection evidence a new region would recompute.


In [4]:
print(k_sweep_quality(data.trainval, B54, tuple(config["k_sweep"]), SEED).to_markdown(index=False))
print()
print("GuardedV-A station majority (full-trainval fit):")
g = routers["GuardedV-A"]
print(pd.DataFrame([{"station": s, "majority": m, "strength": round(g.majority_strength[s], 4)}
                    for s, m in sorted(g.station_majority.items())]).to_markdown(index=False))
print()
print("StationMeanV-B station labels (full-trainval fit):")
m = routers["StationMeanV-B"]
print(pd.DataFrame([{"station": s, "label": lab} for s, lab in sorted(m.station_label.items())]).to_markdown(index=False))


|   K |   silhouette |   calinski_harabasz |   davies_bouldin |   inertia |   mean_station_purity_trainval |   min_cluster_share | cluster_sizes            |
|----:|-------------:|--------------------:|-----------------:|----------:|-------------------------------:|--------------------:|:-------------------------|
|   2 |     0.218834 |             3880.86 |          1.57925 |    623236 |                       1        |            0.272727 | [10624, 3984]            |
|   3 |     0.225664 |             3589.38 |          1.81548 |    528875 |                       0.833006 |            0.264033 | [6513, 3857, 4238]       |
|   4 |     0.187591 |             2855    |          2.01503 |    497221 |                       0.695141 |            0.146632 | [2142, 4053, 3826, 4587] |

GuardedV-A station majority (full-trainval fit):
| station               |   majority |   strength |
|:----------------------|-----------:|-----------:|
| BeaverPass_WA_990     |          0 |          1 |
| Ca

## R3b — Held-out regime assignment per fold
For each LOSO fold, which canonical regime does each router assign to the held-out station's test rows (share in c1 = drier regime)? Single-station test rows make ARI degenerate (0 vs 1), so this table shows the mechanism directly: folds where V0 and Backbone send the held-out station to opposite regimes are the folds that drive the LOSO gap (V0 0.634 vs Backbone 0.617).


In [5]:
assign_rows = []
for station in sorted(data.test["station_id"].unique()):
    tr = data.trainval[data.trainval["station_id"] != station].reset_index(drop=True)
    te = data.test[data.test["station_id"] == station].reset_index(drop=True)
    fr = {n: get_router(STRATEGY_OF[n], backbone_54=B54, v0_features=V0, seed=SEED, tau=TAU).fit(tr) for n in names}
    row = {"held_out": station, "n_te": len(te)}
    for n in names:
        lab = _labs(fr[n], te)
        row[f"{n}_share_c1"] = round(float((lab == 1).mean()), 4)
        row[f"{n}_all_c{int(np.bincount(lab).argmax())}"] = True
    assign_rows.append(row)
cols = ["held_out", "n_te"] + [f"{n}_share_c1" for n in names]
print(pd.DataFrame(assign_rows)[cols].to_markdown(index=False))
print()
print("c1 = drier regime (lower fit-frame mean SMAP_sm_pm_interp_rollmean30, canonicalized per fold)")


| held_out              |   n_te |   V0_Full_share_c1 |   Backbone_share_c1 |   GuardedV-A_share_c1 |   StationMeanV-B_share_c1 |
|:----------------------|-------:|-------------------:|--------------------:|----------------------:|--------------------------:|
| BeaverPass_WA_990     |    626 |             0      |              0      |                0      |                    0      |
| CayusePass_WA         |   1081 |             0      |              0.0139 |                0.0139 |                    0.0139 |
| Darrington            |    999 |             0      |              0      |                0      |                    0      |
| Paradise_WA           |   1067 |             0      |              0.0009 |                0.0009 |                    0.0009 |
| Quinault              |   1044 |             0      |              0      |                0      |                    0      |
| SourdoughGulch_WA_985 |    906 |             1      |              1      |             

## T1/L1 — Model training results (slurm job 2155747)
Reads the training-run CSVs produced by `run_temporal.py` / `run_loso.py` (90 temporal + 105 LOSO jobs, 2500-tree experts, same hyperparameters as formal-eval-1.0). This cell is the sole generator of README §T1/§L1. Formal references: V0_Full (0,0) temporal 0.8118 ± 0.0014 and LOSO 0.6372 from `derived_8.4-formal-eval-1.0/README.md`.


In [6]:
from scipy import stats as _stats

t = pd.read_csv(EXP_DIR / "temporal_seed_summary.csv")
print("### T1 temporal seed-level summary (30 seeds, pooled test R2/RMSE/MAE/bias)")
rows = []
for cid, g in t.groupby("config_id"):
    rows.append({"config": cid, "n": len(g), "mean_R2": round(g.r2.mean(), 6),
                 "std_R2": round(g.r2.std(), 6), "mean_RMSE": round(g.rmse.mean(), 6),
                 "mean_MAE": round(g.mae.mean(), 6), "mean_bias": round(g.bias.mean(), 6)})
print(pd.DataFrame(rows).to_markdown(index=False))
ga = t[t.config_id == "Guarded_Backbone54_k2_c0_0_c1_0"].sort_values("seed")
bb = t[t.config_id == "Clustering_Backbone54_k2_c0_0_c1_0"].sort_values("seed")
d = (ga.r2.to_numpy() - bb.r2.to_numpy())
print(f"Guarded-Backbone paired R2 diff: {d.mean():+.6f}, better on {(d > 0).sum()}/{len(d)} seeds, "
      f"paired t p={_stats.ttest_rel(ga.r2, bb.r2).pvalue:.2e}")
print(f"Guarded vs formal V0_Full (0,0) 0.8118: {ga.r2.mean() - 0.8118:+.6f} (W2 bar: +-0.003)")

l = pd.read_csv(EXP_DIR / "loso_seed_station.csv")
print()
print("### L1 LOSO summary (5 seeds x 7 stations)")
lrows = []
for cid in sorted(l.config_id.unique()):
    sub = l[l.config_id == cid]
    per_seed = sub.groupby("seed").r2.mean()
    lrows.append({"config": cid, "loso_mean_R2": round(per_seed.mean(), 4),
                  "seed_means": [round(float(x), 4) for x in per_seed]})
print(pd.DataFrame(lrows).to_markdown(index=False))
g = l[l.config_id == "Guarded_Backbone54_k2_c0_0_c1_0"].pivot_table(index=["station", "seed"], values="r2", aggfunc="mean")
b = l[l.config_id == "Clustering_Backbone54_k2_c0_0_c1_0"].pivot_table(index=["station", "seed"], values="r2", aggfunc="mean")
dd = (g - b).reset_index()
print("Guarded-Backbone per-held-out-station mean R2 diff (5 seeds):")
print(dd.groupby("station")["r2"].agg(["mean", "std"]).round(4).to_markdown())
print(f"Guarded vs formal V0_Full (0,0) 0.6372: "
      f"{l[l.config_id == 'Guarded_Backbone54_k2_c0_0_c1_0'].groupby('seed').r2.mean().mean() - 0.6372:+.4f} (W2 bar: +-0.010)")


### T1 temporal seed-level summary (30 seeds, pooled test R2/RMSE/MAE/bias)
| config                              |   n |   mean_R2 |   std_R2 |   mean_RMSE |   mean_MAE |   mean_bias |
|:------------------------------------|----:|----------:|---------:|------------:|-----------:|------------:|
| Clustering_Backbone54_k2_c0_0_c1_0  |  30 |  0.811724 | 0.001369 |    0.044201 |   0.033985 |    0.005975 |
| Guarded_Backbone54_k2_c0_0_c1_0     |  30 |  0.811843 | 0.001369 |    0.044187 |   0.033979 |    0.005958 |
| StationMean_Backbone54_k2_c0_0_c1_0 |  30 |  0.811843 | 0.001369 |    0.044187 |   0.033979 |    0.005958 |
Guarded-Backbone paired R2 diff: +0.000119, better on 30/30 seeds, paired t p=1.42e-28
Guarded vs formal V0_Full (0,0) 0.8118: +0.000043 (W2 bar: +-0.003)

### L1 LOSO summary (5 seeds x 7 stations)
| config                              |   loso_mean_R2 | seed_means                               |
|:------------------------------------|---------------:|:------------------